# Récupérer les données géographiques (pays, villes)



In [ ]:
#chargement des modules
import pandas as pd
import datetime
import os
import json
from io import StringIO
import numpy as np
import plotly.express as px
import re

In [ ]:
import s3fs #pour connecter au bucket

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
file_path = BUCKET_OUT + "/" + "Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.jsonl"
print(file_path)
list_id_pub = []
datas =[]
with fs.open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            data = json.loads(line)
            # Extract specific fields
            ids = data.get('all_ids')
            list_id_pub.append(ids[0])
            datas.append(data)
            
        except json.JSONDecodeError:
            continue  # Skip invalid lines

In [ ]:
datas[0]

La cellule ci-dessous récupèrent les données qui nous intéressent depuis le fichier Json. Ces données seront ensuitre structurées dans un dataframe constitué d'une ligne par auteur de chaque publication. Certaines de ces données ne sont en effet pas disponibles dans le fichier "bso-publications-latest_199318270_enriched.csv".


In [ ]:

list_row = []


for n, x in enumerate(list_id_pub):
    
    dict_pub = datas[n]
    if 'hal' in x:
        pub_hal_id = x[3:]
    else:
        pub_hal_id = "no hal id"
        
    if 'authors' in dict_pub:
        #print(n, x)
        
        dict_attribut = {}
        
        for a in dict_pub['authors']:
            #print(a)
            
            if  'full_name' in a:
                full_name = a['full_name']
            else:
                full_name = 'no full_name'
            if 'id_hal_s' in a:
                id_hal_s = a['id_hal_s']
            else:
                id_hal_s = 'no_id_hal_s'
            if 'id_hal_i'  in a:
                id_hal_i = a['id_hal_i']
            else:
                id_hal_i = 'no_id_hal_i'
            if 'orcid' in a:
                orcid = a['orcid']
            else:
                orcid = 'no orcid'
            if "affiliations" in a and len(a["affiliations"]) > 0 :
                list_country = []
                list_ror = []
                list_address = []
                for affil in a["affiliations"]:
                    if "country" in affil:
                        list_country.append(affil["country"])
                    else:
                        list_country.append("no country")
                    if "name" in affil:
                        list_address.append(affil["name"])
                    else:
                        list_address.append("no address")
                    if "ror" in affil:
                        list_ror.append(affil["ror"])
                    else:
                        list_ror.append("no ror")
                country= "|".join(list_country)
                ror= "|".join(list_ror)
                address= "|".join(list_address)
            else:
                country = "no country"
                address = "no address"

            
            
            dict_attribut = {"id": x,
                             "pub_hal_id": pub_hal_id,
                            "id_hal_i":id_hal_i,
                           "id_hal_s": id_hal_s,
                           "full_name": full_name,
                           "orcid": orcid,
                           "country": country,
                           "address": address,
                           "rors": ror}
            list_row.append(dict_attribut)
    

    else:
        dict_attribut = {"id": x,
                             "pub_hal_id": pub_hal_id,
                            "id_hal_i":"no_id_hal_i",
                           "id_hal_s": "no_id_hal_s",
                           "full_name": "no author",
                           "orcid": "no orcid",
                           "country": "no country",
                           "address": "no address",
                           "rors": "no ror"}
        list_row.append(dict_attribut)
        
    

In [ ]:
df0 = pd.DataFrame.from_dict(list_row)
df0

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_author_affil_by_publi.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

